# ROC-AUC curves per combo — GNN(signed)+ChemBERTa, full_full/transductive

Reads the already-finished run `full_full-transductive-gnn_signed-chemberta-tuned250-5x`: reuses the saved cls (GNN) checkpoints (no retraining -- just a forward/encode pass) and the saved per-combo XGBoost boosters, rebuilds each combo's test-set features, and plots a ROC curve per combo (one line per repeat/fold + the mean AUC in the legend).

In [ ]:
import os, pathlib, sys
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

_root = pathlib.Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))
os.chdir(_root)  # orbind.regimes resolves data/... paths relative to cwd, not repo root

from orbind.ensemble import EsmExtractor, GinExtractor
from orbind.gnn_extractor import GnnSignedExtractor
from orbind.regimes import full_full_pairs, load_split

RUN_DIR = _root / "results/ensemble_logs/full_full-transductive-gnn_signed-chemberta-tuned250-5x"
REPEATS = [1, 2, 3, 4, 5]
COMBOS = [("cls",), ("prot",), ("mol",), ("cls", "mol"), ("prot", "mol"), ("cls", "prot", "mol")]

print("run:", RUN_DIR, "exists:", RUN_DIR.exists())

In [ ]:
# Same --source config as the actual run -- must match exactly so checkpoint
# reuse and feature layout line up with the saved boosters.
def build_extractors():
    return {
        "cls": GnnSignedExtractor(
            name="cls",
            protein_path="data/embeddings/proteins/esm1b_650m_mean.npz",
            molecule_path="data/embeddings/molecules/chemberta_77m_m2or.npz",
        ),
        "prot": EsmExtractor(
            name="prot",
            path="data/embeddings/proteins/esm1b_650m_mean.npz",
            model_name="esm1b_t33_650M_UR50S",
        ),
        "mol": GinExtractor(
            name="mol",
            path="data/embeddings/molecules/chemberta_77m_m2or.npz",
            model_name="chemberta_77m",
        ),
    }

pairs = full_full_pairs()
y_all = pairs["label"].to_numpy()
print(f"pairs: {len(pairs)} rows")

In [ ]:
# roc_data[combo][repeat] = (fpr, tpr, auc)
roc_data = {combo: {} for combo in COMBOS}

for repeat in REPEATS:
    checkpoint_dir = RUN_DIR / "checkpoints" / f"repeat_{repeat}"
    train_idx, val_idx, test_idx = load_split("transductive", repeat)
    y_te = y_all[test_idx]

    extractors = build_extractors()
    # cheap thanks to checkpoint reuse: cls loads its 5 saved GNN state_dicts
    # and only does a forward/encode pass, no training.
    Xte_by_name = {
        name: ext.fit_transform(pairs, train_idx, val_idx, test_idx, seed=repeat,
                                 checkpoint_dir=checkpoint_dir)[2]
        for name, ext in extractors.items()
    }

    for combo in COMBOS:
        Xte = np.concatenate([Xte_by_name[n] for n in combo], axis=1)
        booster_path = checkpoint_dir / f"boost_{'+'.join(combo)}.json"
        booster = xgb.Booster()
        booster.load_model(str(booster_path))
        p_te = booster.predict(xgb.DMatrix(Xte))
        fpr, tpr, _ = roc_curve(y_te, p_te)
        roc_data[combo][repeat] = (fpr, tpr, auc(fpr, tpr))

    print(f"repeat {repeat}: done")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)

for ax, combo in zip(axes.flat, COMBOS):
    aucs = []
    for repeat in REPEATS:
        fpr, tpr, auc_val = roc_data[combo][repeat]
        aucs.append(auc_val)
        ax.plot(fpr, tpr, alpha=0.5, lw=1, label=f"fold {repeat} (AUC={auc_val:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.4)
    ax.set_title(f"{'+'.join(combo)}  (mean AUC={np.mean(aucs):.3f} ± {np.std(aucs):.3f})")
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.legend(fontsize=7, loc="lower right")

fig.suptitle("ROC curves per combo -- GNN(signed)+ChemBERTa, full_full/transductive, 5 folds", y=1.02)
fig.tight_layout()
fig.savefig(RUN_DIR / "roc_curves_per_combo.png", dpi=150, bbox_inches="tight")
plt.show()